In [ ]:
from datasets import load_dataset
import torch
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast, Seq2SeqTrainingArguments, Seq2SeqTrainer
from huggingface_hub import login

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

import os
login(token=os.environ["HF_TOKEN"])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

main_seed=42

model_name = "facebook/mbart-large-50"
tokenizer = MBart50TokenizerFast.from_pretrained(model_name)
model = MBartForConditionalGeneration.from_pretrained(model_name)

tokenizer.src_lang = "ru_RU"
tokenizer.tgt_lang = "ru_RU"

dataset = load_dataset(
    "csv", 
    data_files={
        "train": "seq2seq_dataset/train.csv",
        "test": "seq2seq_dataset/test.csv", 
        "val": "seq2seq_dataset/val.csv", 
        },
    delimiter="\t"
)

In [ ]:
def preprocess(batch):
    model_inputs = tokenizer(
        batch["whisper_out"],
        max_length=225,
        truncation=True,
        padding="max_length"
    )

    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            batch["label"],
            max_length=225,
            truncation=True,
            padding="max_length"
        )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized = dataset.map(
    preprocess,
    batched=True,
    num_proc=2,
    remove_columns=dataset["train"].column_names
)

In [ ]:
import evaluate
import re
import unicodedata

def normalize(text: str) -> str:
    text = text.lower()
    
    normalized_chars = []
    for ch in text:
        cat = unicodedata.category(ch)
        if cat.startswith("L") or cat == "Zs":
            normalized_chars.append(ch)
        elif unicodedata.combining(ch) and ch in "̄":
            normalized_chars.append(ch)
        elif ch in ["ӈ"]:
            normalized_chars.append(ch)

    return re.sub(r"\s+", " ", "".join(normalized_chars)).strip()

metric_wer = evaluate.load("wer")
metric_cer = evaluate.load("cer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    orh_wer = 100 * metric_wer.compute(predictions=pred_str, references=label_str)
    orh_cer = 100 * metric_cer.compute(predictions=pred_str, references=label_str)

    pred_str_norm = [normalize(pred) for pred in pred_str]
    label_str_norm = [normalize(label) for label in label_str]

    pred_str_norm = [
        pred_str_norm[i] for i in range(len(pred_str_norm)) if len(label_str_norm[i]) > 0
    ]
    label_str_norm = [ 
        label_str_norm[i] for i in range(len(label_str_norm))if len(label_str_norm[i]) > 0
    ]

    wer = 100 * metric_wer.compute(predictions=pred_str_norm, references=label_str_norm)
    cer = 100 * metric_cer.compute(predictions=pred_str_norm, references=label_str_norm)

    return {"wer": wer, "cer": cer, "orh_wer": orh_wer, "orh_cer": orh_cer}

In [ ]:
args = Seq2SeqTrainingArguments(
    output_dir="./mbart_correction_mansi_v1",
    overwrite_output_dir=True,

    learning_rate=3e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,

    fp16=False,
    fp16_full_eval=False,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,


    num_train_epochs=20,
    eval_strategy="steps",
    eval_steps=2000,
    save_strategy="steps",
    save_steps=2000,
    save_total_limit=2,
    
    seed=main_seed,
    data_seed=main_seed,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    report_to=["tensorboard"],
    push_to_hub=False,
    hub_private_repo=False,
    hub_always_push=False,
    optim="adamw_torch",
    predict_with_generate=True,
    generation_max_length=225
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    compute_metrics=compute_metrics,
    tokenizer=tokenizer
)
trainer.train()

trainer.save_model("./mbart_correction_mansi_v1")